<a href="https://colab.research.google.com/github/yoonwoojeong/yoonwoojeong/blob/main/MDtransformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
import jax
import jax.numpy as jnp
from jax import grad, jit, vmap, random
import matplotlib.pyplot as plt

# 1. Configuration
class Config:
    D_e, a_morse, r0 = 105.0, 2.2, 0.9572
    k_angle, theta0 = 75.0, 1.824
    q_O_core, q_D, q_H = 0.5847, -1.4323, 0.4238
    k_drude, k_e = 1000.0, 332.0636
    sigma_smear = 0.3
    num_molecules = 1
    dt = 0.0005
    seq_len = 32
    embed_dim = 256
    num_layers = 2
    dropout_rate = 0.1
    learning_rate = 1e-4
    weight_decay = 0.01
    epochs = 300
    sim_steps = 3000

cfg = Config()
key = random.PRNGKey(42)

# 2. Refined Physics (used to generate initial 'environment' for self-supervision)
def compute_forces(pos, drude_pos, cfg):
    O, H1, H2 = pos[0, 0], pos[0, 1], pos[0, 2]
    D = drude_pos[0]
    def morse_force(p1, p2):
        r_vec = p1 - p2
        r = jnp.linalg.norm(r_vec) + 1e-6
        f_mag = 2 * cfg.D_e * cfg.a_morse * (1 - jnp.exp(-cfg.a_morse*(r-cfg.r0))) * jnp.exp(-cfg.a_morse*(r-cfg.r0))
        return (f_mag / r) * r_vec
    f_OH1, f_OH2 = morse_force(O, H1), morse_force(O, H2)
    v1, v2 = H1 - O, H2 - O
    theta = jnp.arccos(jnp.clip(jnp.dot(v1, v2) / (jnp.linalg.norm(v1) * jnp.linalg.norm(v2) + 1e-6), -1.0, 1.0))
    f_angle = (-cfg.k_angle * (theta - cfg.theta0)) * (v1 + v2) * 0.1
    def smeared_coulomb(p1, p2, q1, q2):
        r_vec = p1 - p2
        r = jnp.linalg.norm(r_vec) + 1e-6
        val = r / (jnp.sqrt(2) * cfg.sigma_smear)
        f_mag = (cfg.k_e * q1 * q2) * (jax.scipy.special.erf(val) - (2*val/jnp.sqrt(jnp.pi))*jnp.exp(-val**2)) / (r**2)
        return (f_mag / r) * r_vec
    all_p, qs = jnp.array([O, H1, H2, D]), jnp.array([cfg.q_O_core, cfg.q_H, cfg.q_H, cfg.q_D])
    f_nb = jnp.zeros((4, 3))
    for i in range(4):
        for j in range(4):
            if i != j: f_nb = f_nb.at[i].add(smeared_coulomb(all_p[i], all_p[j], qs[i], qs[j]))
    f_ds = -cfg.k_drude * (D - O)
    fa = jnp.zeros((1, 3, 3)).at[0,0].set(f_nb[0]-f_OH1-f_OH2-f_ds).at[0,1].set(f_nb[1]+f_OH1+f_angle).at[0,2].set(f_nb[2]+f_OH2-f_angle)
    return fa, (f_nb[3] + f_ds)[None, :]

def extract_features(pos, drude_pos, cfg):
    O, H1, H2 = pos[0, 0], pos[0, 1], pos[0, 2]
    D = drude_pos[0]
    dipole = cfg.q_O_core*O + cfg.q_H*H1 + cfg.q_H*H2 + cfg.q_D*D
    r_OH1, r_OH2, r_HH = jnp.linalg.norm(O-H1), jnp.linalg.norm(O-H2), jnp.linalg.norm(H1-H2)
    return jnp.concatenate([dipole, jnp.array([r_OH1, r_OH2, r_HH]), pos.flatten()])

# 3. Transformer Model Architecture
def init_transformer(key, in_dim, e_dim):
    k1, k2, k3 = random.split(key, 3)
    def get_layer(k):
        sk = random.split(k, 6)
        return [[random.normal(sk[i], (e_dim, e_dim)) * jnp.sqrt(2/e_dim) for i in range(4)],
                [random.normal(sk[4], (e_dim, e_dim*4)) * jnp.sqrt(2/e_dim), jnp.zeros(e_dim*4),
                 random.normal(sk[5], (e_dim*4, e_dim)) * jnp.sqrt(2/(e_dim*4)), jnp.zeros(e_dim)]]
    layers = [get_layer(random.split(k2, cfg.num_layers)[i]) for i in range(cfg.num_layers)]
    return [random.normal(k1, (in_dim, e_dim)) * 0.02, layers, random.normal(k3, (e_dim, in_dim)) * 0.02]

def layer_norm(x): return (x - jnp.mean(x, -1, keepdims=True)) / jnp.sqrt(jnp.var(x, -1, keepdims=True) + 1e-6)

def forward(params, seq):
    x = jnp.dot(seq, params[0])
    for attn, ff in params[1]:
        q, k, v = jnp.dot(x, attn[0]), jnp.dot(x, attn[1]), jnp.dot(x, attn[2])
        res = jnp.dot(jax.nn.softmax(q @ k.T / jnp.sqrt(x.shape[-1])) @ v, attn[3])
        x = layer_norm(x + res)
        res_ff = jnp.dot(jax.nn.gelu(jnp.dot(x, ff[0]) + ff[1]), ff[2]) + ff[3]
        x = layer_norm(x + res_ff)
    return jnp.dot(x, params[2])

# 4. Self-Supervised Logic: Predict next from internal state consistency
@jit
def train_step(p, m, v, t, batch_seq):
    # The target is the sequence shifted by one (Self-Supervised Next-Step Prediction)
    def loss_fn(p):
        preds = vmap(forward, (None, 0))(p, batch_seq)
        # Loss on predicting the next feature set in the sequence
        return jnp.mean((preds[:, :-1] - batch_seq[:, 1:])**2)

    grads = grad(loss_fn)(p)
    t += 1
    m = jax.tree_util.tree_map(lambda x,y: 0.9*x+0.1*y, m, grads)
    v = jax.tree_util.tree_map(lambda x,y: 0.999*x+0.001*y**2, v, grads)
    p = jax.tree_util.tree_map(lambda p,mh,vh: p-cfg.learning_rate*(mh/(1-0.9**t)/(jnp.sqrt(vh/(1-0.999**t))+1e-8)+cfg.weight_decay*p), p, m, v)
    return p, m, v, t, loss_fn(p)

# 5. Run the Self-Supervised Experiment
pos = jnp.array([[[0,0,0], [0.9,0.6,0], [-0.9,0.6,0]]])
drude_pos = pos[:,0,:] + 0.005
vel, initial_traj = random.normal(key, (1, 3, 3)) * 0.001, []
for _ in range(cfg.sim_steps):
    fa, fd = compute_forces(pos, drude_pos, cfg)
    vel += 0.5 * fa * cfg.dt; pos += vel * cfg.dt; drude_pos += fd * 0.001
    initial_traj.append(extract_features(pos, drude_pos, cfg))

initial_traj = jnp.array(initial_traj)
batch_data = jnp.array([initial_traj[i:i+cfg.seq_len] for i in range(0, len(initial_traj)-cfg.seq_len, 10)])

params = init_transformer(key, initial_traj.shape[-1], cfg.embed_dim)
m, v, t = jax.tree_util.tree_map(jnp.zeros_like, params), jax.tree_util.tree_map(jnp.zeros_like, params), 0

print("Self-Supervised Training on manifold dynamics...")
for ep in range(cfg.epochs):
    params, m, v, t, loss = train_step(params, m, v, t, batch_data)
    if ep % 50 == 0: print(f"Epoch {ep}, Self-Supervised Loss: {loss:.6f}")

# 6. Autoregressive Generation (Self-Propagated Simulation)
seed_seq = initial_traj[:cfg.seq_len]
generated_traj = [seed_seq[-1]]
curr_seq = seed_seq

for _ in range(500):
    next_step = forward(params, curr_seq)[-1] # Predict next
    generated_traj.append(next_step)
    curr_seq = jnp.concatenate([curr_seq[1:], next_step[None, :]], axis=0)

generated_traj = jnp.array(generated_traj)
true_pos = initial_traj[cfg.seq_len:cfg.seq_len+501, -9:].reshape(-1, 3, 3)
gen_pos = generated_traj[:, -9:].reshape(-1, 3, 3)

def get_dist(p): return jnp.linalg.norm(p[:, 0, :] - p[:, 1, :], axis=-1)

plt.figure(figsize=(10, 4))
plt.plot(get_dist(true_pos), label='Physics Engine (GT)', alpha=0.7)
plt.plot(get_dist(gen_pos), label='Self-Supervised Transformer', linestyle='--')
plt.title("Autoregressive Generation of O-H Bond Dynamics"); plt.legend(); plt.show()

KeyboardInterrupt: 